In [4]:
import pandas as pd
import numpy as np

# Chargement du dataset
chemin_fichier = "../../data/raw/rappelconso-v2.csv"

df = pd.read_csv(chemin_fichier, sep=";")

# Nettoyage des noms de colonnes
df.columns = df.columns.str.replace('\ufeff', '', regex=False)

df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace("'", '_')
    .str.normalize('NFKD')
    .str.encode('ascii', errors='ignore')
    .str.decode('utf-8')
)

print("Dataset chargé :", df.shape)

Dataset chargé : (19258, 34)


In [ ]:
# Préparation de l'exploration pour le clustering

print("DIMENSIONS DU DATASET")
print(df.shape)

print("\nNOMBRE DE VALEURS DISTINCTES PAR VARIABLE")
display(
    df.nunique()
    .sort_values(ascending=False)
    .to_frame(name="Nombre_de_valeurs_distinctes")
)

DIMENSIONS DU DATASET
(19258, 34)

NOMBRE DE VALEURS DISTINCTES PAR VARIABLE


,Nombre_de_valeurs_distinctes
numero_fiche,19258
lien_vers_affichette_pdf,19258
rappel_guid,19258
id,19258
lien_vers_la_fiche_rappel,19258
date_publication,18606
identification_produits,17581
liens_vers_les_images,17315
modeles_ou_references,17200
libelle,15729


## Regroupement des rappels par motif : normalisation, TF-IDF et K-means

On harmonise les accents, la ponctuation et quelques variantes orthographiques explicitement listées (listeria, salmonelle, *E. coli*). Les formulations normalisées identiques sont vectorisées une seule fois ; les étiquettes sont ensuite réattribuées à tous les rappels. Ainsi, les doublons très fréquents ne dominent pas l'apprentissage, mais les tailles de groupes représentent bien le nombre de rappels. La normalisation est volontairement prudente : elle ne remplace pas automatiquement « détection », « suspicion » et « présence », qui peuvent porter des nuances différentes.

La comparaison teste plusieurs valeurs de *k*. Le score de silhouette cosinus est calculé sur un échantillon de formulations distinctes ; il aide à comparer les essais, mais doit être interprété avec les exemples de chaque groupe.

In [3]:
import re
import unicodedata

from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score

# Harmonisation reproductible des accents, ponctuations et variantes courantes.
def normaliser_motif(motif):
    texte = unicodedata.normalize("NFKD", str(motif).lower())
    texte = "".join(caractere for caractere in texte if not unicodedata.combining(caractere))
    texte = re.sub(r"\b(?:liesteria|lysteria|listerias?)\b", "listeria", texte)
    texte = re.sub(r"\b(?:samonelles?|salmonell(?:a|e)s?)\b", "salmonelle", texte)
    texte = re.sub(r"\b(?:e[\s.:-]*coli|ecoli)\b", "escherichia coli", texte)
    texte = re.sub(r"[^a-z0-9]+", " ", texte)
    return re.sub(r"\s+", " ", texte).strip()

motifs_bruts = df["motif_rappel"].fillna("").astype(str).str.strip()
motifs_normalises = motifs_bruts.map(normaliser_motif)
masque_motifs = motifs_normalises.ne("")
formulations_uniques = pd.Index(motifs_normalises.loc[masque_motifs].unique())

print(f"Rappels dans le dataset : {len(df):,}")
print(f"Motifs non vides : {int(masque_motifs.sum()):,}")
print(f"Formulations distinctes avant normalisation : {motifs_bruts.loc[masque_motifs].nunique():,}")
print(f"Formulations distinctes après normalisation : {len(formulations_uniques):,}")

stopwords_fr = """a au aux avec ce ces dans de des du elle en et eux il ils je la le les leur lui ma mais me mes moi mon ne nos notre nous on ou par pas pour qu que qui sa se ses son sur ta te tes toi ton tu un une vos votre vous c d l m n s t y est sont etre ete avoir fait faire produit produits rappel rappels""".split()
vectoriseur_motifs = TfidfVectorizer(
    stop_words=stopwords_fr,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
)
X_motifs = vectoriseur_motifs.fit_transform(formulations_uniques)
print(f"Matrice TF-IDF : {X_motifs.shape[0]:,} formulations × {X_motifs.shape[1]:,} termes")

# On compare les mêmes formulations pour chaque valeur de k.
valeurs_k = [4, 6, 8, 10, 12]
resultats_k = {}
lignes_scores = []
lignes_tailles = []
for k in valeurs_k:
    modele = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    etiquettes = modele.fit_predict(X_motifs)
    silhouette = silhouette_score(
        X_motifs,
        etiquettes,
        metric="cosine",
        sample_size=min(1500, X_motifs.shape[0]),
        random_state=42,
    )
    resultats_k[k] = (modele, etiquettes, silhouette)
    etiquettes_formulations = pd.Series(etiquettes, index=formulations_uniques)
    etiquettes_rappels = motifs_normalises.map(etiquettes_formulations)
    tailles = etiquettes_rappels.loc[masque_motifs].value_counts().sort_index()
    lignes_scores.append({"k": k, "silhouette_cosinus": round(silhouette, 4)})
    for groupe, taille in tailles.items():
        lignes_tailles.append({
            "k": k,
            "groupe": int(groupe),
            "nombre_de_rappels": int(taille),
            "part_des_rappels": f"{taille / int(masque_motifs.sum()):.1%}",
        })

print("\nComparaison des scores :")
display(pd.DataFrame(lignes_scores))
print("Tailles détaillées des groupes pour chaque k :")
display(pd.DataFrame(lignes_tailles))

# Le meilleur score sert à choisir le modèle à inspecter, pas à conclure seul.
k_retenu = max(resultats_k, key=lambda valeur: resultats_k[valeur][2])
modele_retenu, etiquettes_uniques, score_retenu = resultats_k[k_retenu]
etiquettes_par_formulation = pd.Series(etiquettes_uniques, index=formulations_uniques)
df["cluster_motif"] = motifs_normalises.map(etiquettes_par_formulation).astype("Int64")

termes = vectoriseur_motifs.get_feature_names_out()
occurrences = pd.DataFrame({
    "cluster": df.loc[masque_motifs, "cluster_motif"].astype(int).to_numpy(),
    "motif_normalise": motifs_normalises.loc[masque_motifs].to_numpy(),
})

print(f"\nInspection du modèle k={k_retenu} (silhouette cosinus={score_retenu:.4f})")
for cluster_id in range(k_retenu):
    effectif = int((occurrences["cluster"] == cluster_id).sum())
    nb_formulations = int((etiquettes_uniques == cluster_id).sum())
    mots_cles = termes[modele_retenu.cluster_centers_[cluster_id].argsort()[-8:][::-1]]
    exemples = (
        occurrences.loc[occurrences["cluster"] == cluster_id, "motif_normalise"]
        .value_counts()
        .head(5)
    )
    print(f"\nGroupe {cluster_id} — {effectif:,} rappels ({effectif / int(masque_motifs.sum()):.1%}), {nb_formulations:,} formulations")
    print("Termes clés : " + ", ".join(mots_cles))
    print("Motifs fréquents :")
    for motif, nombre in exemples.items():
        print(f"  • {int(nombre):>4} rappel(s) — {motif[:180]}")

print("\nLes étiquettes sont disponibles dans df['cluster_motif'] (valeur manquante si motif vide).")

Rappels dans le dataset : 19,258
Motifs non vides : 19,257
Formulations distinctes avant normalisation : 9,657
Formulations distinctes après normalisation : 8,915


Matrice TF-IDF : 8,915 formulations × 16,364 termes



Comparaison des scores :


,k,silhouette_cosinus
0,4,0.0193
1,6,0.0297
2,8,0.0378
3,10,0.0371
4,12,0.0398


Tailles détaillées des groupes pour chaque k :


,k,groupe,nombre_de_rappels,part_des_rappels
0,4,0,2285,11.9%
1,4,1,10315,53.6%
2,4,2,3307,17.2%
3,4,3,3350,17.4%
4,6,0,10590,55.0%
5,6,1,2851,14.8%
6,6,2,469,2.4%
7,6,3,459,2.4%
8,6,4,1553,8.1%
9,6,5,3335,17.3%



Inspection du modèle k=12 (silhouette cosinus=0.0398)

Groupe 0 — 160 rappels (0.8%), 139 formulations
Termes clés : bouche, etouffer, mettre, bouche etouffer, mettre bouche, facilement, enfant, petit enfant
Motifs fréquents :
  •   10 rappel(s) — l emballage n est pas a l epreuve des enfants et les piles bouton sont donc facilement accessibles aux enfants un enfant peut les mettre dans la bouche et s etouffer les piles bout
  •    6 rappel(s) — le compartiment a piles peut facilement etre ouvert sans utiliser d outils et les piles bouton a l interieur sont facilement accessibles un enfant peut les mettre dans la bouche et
  •    6 rappel(s) — les poppers peuvent etre facilement detaches creant de petites pieces un petit enfant peut les mettre dans la bouche et s etouffer en outre les poppers destines aux fixations ne so
  •    3 rappel(s) — le materiau de rembourrage fibreux du jouet est facilement accessible un petit enfant pourrait mettre du materiel de rembourrage dans la bouche e